In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from anthropic import Anthropic

client = Anthropic()
model = os.getenv("CLAUDE_MODEL")

In [4]:
def add_user_message(messages, text):
	user_message = {"role": "user", "content": text}
	messages.append(user_message)


def add_assistant_message(messages, text):
	assistant_message = {"role": "assistant", "content": text}
	messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0):
	params = {
		"model" : model,
		"max_tokens" : 100,
		"messages" : messages,
		"temperature" : temperature
	}

	if system:
		params["system"] = system
	
	message = client.messages.create(**params)

	return message.content[0].text


In [ ]:
"""
print(event) sends back:
 1. RawMessageStartEvent -- a new message is being sent
 2. RawContentBlockStartEvent -- start of a new block, which contains a text message, tool use, or other
 3. RawContentBlockDeltaEvent -- chunk related to the latest content block that was started
 4. RawContentBlockStopEvent -- the current content block has been completed
 5. RawMessageDeltaEvent -- the current message is complete
 6. RawMessageStopEvent -- end of info about the current message

We care most about the RawContentBlockDeltaEvent, since this cotnains the text being generated by Claude and sent back to he server chunk by chunk
 - It contains a sequence of "ContentBlockDelta", each of which returns text
"""


messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
	model=model,
	max_tokens=100,
	messages=messages,
	stream=True
)

for event in stream:
  print(event)

**Streaming w/ Anthropic SDK:**

In [ ]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
  model=model,
  max_tokens=1000,
  messages=messages
) as stream:
	for text in stream.text_stream:
		# print(text, end="")
		pass

stream.get_final_message() # this gets all individual text chunks and assembles them together. Useful for placing the text in a database (like our mock messages[])

ParsedMessage(id='msg_012sj9CFgNJXhn9hSy664Qwq', container=None, content=[ParsedTextBlock(citations=None, text='# FakeDB\n\nFakeDB is an in-memory database system that generates and stores randomly populated tables with synthetic data for rapid prototyping and testing without requiring actual user information or production datasets.', type='text', parsed_output=None)], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=44, server_tool_use=None, service_tier='standard'))